# Stochastic heat equation — statistical analysis

Two statistical experiments on the spectral solver of `src/heat.py`, which
projects the initial profile onto the sine basis, evolves each modal
coefficient as a decoupled Ornstein–Uhlenbeck process, and transforms back
(report §11):

1. **Monte-Carlo statistics** — the empirical mean and $\pm1,\pm2$
   standard-deviation bands of $u(x)$ at two fixed times.
2. **Covariance comparison** — the *exact* one-step Ornstein–Uhlenbeck noise
   variance against a naive flat $\sigma^2\,\mathrm{d}t$ variance, on a matched
   noise stream.

In [ ]:
import sys
from pathlib import Path

# Locate the project root (the folder containing src/) so the solver modules
# import cleanly and figures are written to figures/.
ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
FIGURES = ROOT / "figures"

In [ ]:
import math

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib import cm
from mpl_toolkits.mplot3d import axes3d  # noqa: F401  (registers 3d projection)

import heat

## 1. Monte-Carlo mean and standard deviation

We drive `heat.heat_spectral` with a smooth bump initial profile, constant
noise modulation $g\equiv1$, and the identity covariance $Q=I$ (space–time
white noise — every mode kicked equally), then collect many independent runs.

In [ ]:
K_modes = 50

def u0(x):
    return 4 * x * (1 - x)   # smooth bump, zero at both ends

def g(t):
    return 1                 # constant-in-time noise scaling

Q = np.identity(K_modes)     # space-time white noise

In [ ]:
rng = np.random.default_rng(67)

T, Nt, Nx = 5.0, 1000, 50    # run to t = 5 so both t = 2 and t = 5 fall inside one run
N = 300                      # number of Monte-Carlo samples

t_grid = np.linspace(0, T, Nt)
i2 = np.argmin(np.abs(t_grid - 2))   # time-row nearest t = 2
i5 = np.argmin(np.abs(t_grid - 5))   # time-row nearest t = 5

Run `N` independent simulations and record the spatial profile at $t=2$ and
$t=5$. (With `N = 300` this takes a couple of minutes; lower `N` for a quick
check.)

In [ ]:
samples_t2 = np.zeros((N, Nx + 1))
samples_t5 = np.zeros((N, Nx + 1))

for i in range(N):
    _, _, u = heat.heat_spectral(u0, T, Nt, Nx, K_modes, rng, d=0.01, var=0.5, g=g, Q=Q)
    samples_t2[i] = u[i2]
    samples_t5[i] = u[i5]

In [ ]:
mean_t2 = samples_t2.mean(axis=0)
std_t2 = samples_t2.std(axis=0, ddof=1)

mean_t5 = samples_t5.mean(axis=0)
std_t5 = samples_t5.std(axis=0, ddof=1)

The empirical mean (blue) sits inside its $\pm1$ (green) and $\pm2$ (red)
standard-deviation bands. By $t=5$ the mean has diffused toward zero while the
noise has built up a roughly uniform spread across the domain.

In [ ]:
x = np.linspace(0, 1, Nx + 1)
fig, axs = plt.subplots(1, 2, figsize=(13, 5), sharey=True)

for ax, mean, std, tt in [(axs[0], mean_t2, std_t2, 2), (axs[1], mean_t5, std_t5, 5)]:
    ax.plot(x, mean, 'b-', lw=2, label='mean')
    ax.plot(x, mean + std, 'g--', lw=1.4, label='+/- 1 std')
    ax.plot(x, mean - std, 'g--', lw=1.4)
    ax.plot(x, mean + 2 * std, 'r:', lw=1.4, label='+/- 2 std')
    ax.plot(x, mean - 2 * std, 'r:', lw=1.4)
    ax.axhline(0, color='k', lw=0.5)
    ax.set(xlabel='x', title=f't = {tt}')
    ax.legend()

axs[0].set_ylabel('u')
fig.suptitle(f'Distribution of u(x) at two times  (N={N} runs, u0=bump, Q=identity)')
fig.tight_layout()
fig.savefig(FIGURES / "m2r_statistics.png", dpi=150, bbox_inches="tight")
plt.show()

## 2. Exact vs naive noise covariance

When the modal SDEs are integrated exactly over a step, the noise kick for mode
$k$ has variance

$$q_k = \text{var}\cdot\frac{1 - e^{-2\lambda_k\,\mathrm{d}t}}{2\lambda_k},\qquad \lambda_k = d\,(k\pi)^2,$$

rather than the naive flat value $\text{var}\cdot\mathrm{d}t$ used above
(`heat.ou_step_variance` and `heat.flat_step_variance`). We run
`heat.heat_spectral_diag` with both choices on the *same* noise stream — a
shared seed — and compare the resulting surfaces.

In [ ]:
def init(x):
    return 5 * x * (x - 1) * math.sin(23 * x)   # wiggly initial profile

def g(t):
    return 1

In [ ]:
T, Nt, Nx, K_modes, d, var = 5, 1000, 50, 50, 0.01, 0.5
seed = 67
dt = T / Nt

X, Y, u_exact = heat.heat_spectral_diag(init, T, Nt, Nx, np.random.default_rng(seed),
                                        K_modes, d, g, heat.ou_step_variance(K_modes, d, dt, var))
_, _, u_ident = heat.heat_spectral_diag(init, T, Nt, Nx, np.random.default_rng(seed),
                                        K_modes, d, g, heat.flat_step_variance(K_modes, d, dt, var))
u_diff = u_exact - u_ident

The exact and naive surfaces look almost identical; their difference (right
panel, note the much smaller vertical scale) is tiny, confirming that for these
parameters the flat $\text{var}\cdot\mathrm{d}t$ approximation is a reasonable
stand-in for the exact OU step.

In [ ]:
fig, axs = plt.subplots(ncols=3, subplot_kw=dict(projection='3d'), figsize=(16, 5))
light = mcolors.LightSource(270, 45)

def show(ax, u, title, zlim):
    rgb = light.shade(u.T, cmap=cm.gist_earth, vert_exag=0.1, blend_mode='soft')
    ax.plot_surface(X, Y, u.T, rstride=1, cstride=1, facecolors=rgb,
                    linewidth=0, antialiased=False, shade=False)
    ax.set(xlim=(0, T), ylim=(0, 1), zlim=zlim, xlabel='T', ylabel='X', zlabel='H')
    ax.set_title(title)

show(axs[0], u_exact, "exact (OU covariance)",      (-2.5, 2.5))
show(axs[1], u_ident, "identity (flat var*dt)",     (-2.5, 2.5))
show(axs[2], u_diff,  "difference (exact - ident)", (-0.5, 0.5))

fig.tight_layout()
fig.savefig(FIGURES / "m2r_covariance.png", dpi=120, bbox_inches="tight")
plt.show()
print("max|exact - identity| =", np.abs(u_diff).max())